In [1]:
from regex_labels.patterns_and_extractors import (
    extract_first_appearance_hearing, determine_bond_status, should_include_ror,
    extract_the_court_appointed_the_orleans_public_defender, extract_ankle_monitoring,
    extract_domestic_stay_away_order, determine_charge_status, extract_gwens_law_hearing,
    has_probable_cause, extract_defendant_in_custody_and_not_in_court,
    extract_the_state_was_not_present, extract_pre_trial_conference,
    determine_continuance, plea_status, guilty_or_no_guilty, sentence_jail_status
)

Config initialized with device: cuda


In [2]:
from tqdm.notebook import tqdm
import pandas as pd
# Load the data
current_clerks_proceedings = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_current.xlsx"
clerks = pd.read_excel(current_clerks_proceedings, sheet_name=None)

# Filter for MAG data only
mag_dictionary = {}
for clerk, df in clerks.items():
    df_for_clerk = df.copy()
    # Filter the dataframe for identifier column where value is 'mag'
    mag_df = df_for_clerk[df_for_clerk['identifier'] == 'mag']
    mag_dictionary[clerk] = mag_df

print(f"Initial MAG clerks: {len(mag_dictionary)}")

# Remove any clerks from the dictionary where the dataframe is empty
mag_dictionary = {k: v for k, v in mag_dictionary.items() if not v.empty}
print(f"MAG clerks with data: {len(mag_dictionary)}")

Initial MAG clerks: 158
MAG clerks with data: 78


In [3]:
def add_columns_to_dataframe(df):
    df = df.copy()
    
    df['first_appearance_hearing'] = df['text'].apply(extract_first_appearance_hearing)
    df['bond_status'] = df['text'].apply(determine_bond_status)
    df['include_ror'] = df['text'].apply(should_include_ror)
    df['court_appointed_orleans_public_defender'] = df['text'].apply(extract_the_court_appointed_the_orleans_public_defender)
    df['ankle_monitoring'] = df['text'].apply(extract_ankle_monitoring)
    df['domestic_stay_away_order'] = df['text'].apply(extract_domestic_stay_away_order)
    df['charge_status'] = df['text'].apply(determine_charge_status)
    df['gwens_law_hearing'] = df['text'].apply(extract_gwens_law_hearing)
    df['has_probable_cause'] = df['text'].apply(has_probable_cause)
    df['defendant_in_custody_and_not_in_court'] = df['text'].apply(extract_defendant_in_custody_and_not_in_court)
    df['state_was_not_present'] = df['text'].apply(extract_the_state_was_not_present)
    df['pre_trial_conference'] = df['text'].apply(extract_pre_trial_conference)
    df['continuance'] = df['text'].apply(determine_continuance)
    df['plea_status'] = df['text'].apply(plea_status)
    df['guilty_or_no_guilty'] = df['text'].apply(guilty_or_no_guilty)
    df['sentence_jail_status'] = df['text'].apply(sentence_jail_status)

    return df

In [4]:
from tqdm.notebook import tqdm
# Apply feature extraction to each MAG dataframe
print("Processing MAG data...")
for clerk, df in tqdm(mag_dictionary.items(), position=0):
    mag_dictionary[clerk] = add_columns_to_dataframe(df)
print("MAG processing complete!")

Processing MAG data...


  0%|          | 0/78 [00:00<?, ?it/s]

MAG processing complete!


In [5]:
#if a cell has either 'NONE', 'NO_DECISION', or 'NO_SENTENCE_WAS_IMPOSED' for any of the new columns put it into NAs
for clerk, df in tqdm(mag_dictionary.items()):
    mag_dictionary[clerk] = df.replace(['NONE', 'NO_DECISION', 'NO_SENTENCE_WAS_IMPOSED'], pd.NA)
    #remove the case_var and date columns
    mag_dictionary[clerk] = mag_dictionary[clerk].drop(columns=['case_var', 'date'])
    #drop any duplicate rows
    mag_dictionary[clerk] = mag_dictionary[clerk].drop_duplicates()
    


  0%|          | 0/78 [00:00<?, ?it/s]

In [6]:
mag_dictionary_2 = mag_dictionary.copy()

In [7]:
import re
date_pattern = re.compile(
    r'\b('
        # ISO-like: 2025-09-17 or 2025.09.17 or 2025/09/17
        r'\d{4}[-./]\d{1,2}[-./]\d{1,2}'
        r'|'
        # US-style: 09/16/2025 or 9/6/25
        r'\d{1,2}[-/]\d{1,2}[-/]\d{2,4}'
        r'|'
        # Day first: 17.09.2025 or 17-09-25
        r'\d{1,2}[-.]\d{1,2}[-.]\d{2,4}'
        r'|'
        # Month name first: September 15, 2025
        r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec|January|February|March|April|June|July|August|September|October|November|December)'
        r'\s+\d{1,2},?\s+\d{2,4}'
        r'|'
        # Day first with month name: 15 September 2025 or 15 Sept 25
        r'\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec|January|February|March|April|June|July|August|September|October|November|December)\s+\d{2,4}'
    r')\b',
    flags=re.IGNORECASE
)
def remove_dates(text):
    return date_pattern.sub('[DATE]', text)

mag_dictionary_2 = {clerk: df.assign(text=df['text'].apply(remove_dates)) for clerk, df in mag_dictionary_2.items()}

In [8]:
for clerk, df in tqdm(mag_dictionary_2.items()):
    mag_dictionary_2[clerk] = mag_dictionary_2[clerk].drop_duplicates()

  0%|          | 0/78 [00:00<?, ?it/s]

In [9]:
#make it into a xlsx file with each clerk as a sheet
with pd.ExcelWriter('mag_clerks_with_labels.xlsx') as writer:
    for clerk, df in mag_dictionary_2.items():
        df.to_excel(writer, sheet_name=clerk, index=False)